# Week 2 — The model is just a rule you can read

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/notebooks/02_your_first_readable_model.ipynb)

You'll:
1. Write a **1-line hand rule** and rank pages with it.
2. Fit a **depth-2 decision tree** and `print` it — the model "learned" a readable if/else. Then compare — where does it beat your rule, and where doesn’t it?
3. See **why you never feed the outcome back in** — that's leakage.

The payoff isn't a high score. It's: *my intuition was rough, the model found the real signal, and I can read exactly what it found.*

## 0. Setup (Colab or local)

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

## 1. A rule you write by hand: `stale x visible`
Intuition: a page worth reviewing is one that is **stale** (not updated in a while) **and** still **visible** (getting impressions). Rank those by how much exposure they have.

In [ ]:
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

top10 = df.sort_values("hand_rule_score", ascending=False).head(10)
top10[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]]

We need a way to score any ranking. **Precision@K** = of the top K pages a ranking flags, what fraction are actually declining.

In [ ]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")

## 2. Let a model learn the rule — then read it
A **depth-2 decision tree** can only ask 3 yes/no questions. That constraint is the point: whatever it learns, you can read.

We give it a few **pre-decision** signals — never product flags.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

That printout **is** the model — a human-readable if/else. Now rank pages by the tree's probability and score it the same way.

In [ ]:
tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

Look closely: the tree **wins at Precision@50** but your hand rule **wins at Precision@20**. Both results are real. A sharp human rule can be excellent at the very top of the list; the model's advantage shows up deeper, where simple rules run out of signal. Saying exactly that — instead of "the model is better" — is what honest evaluation sounds like.

## 3. Why you can't feed the outcome back in
Your label is `trend_direction == "down"`, and `trend_pct` is the exact percentage change that bucket is computed from — so it **is** the answer in disguise. Watch what happens if you feed it in as a feature:

In [ ]:
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
print(f"'Leaky' tree Precision@50: {precision_at_k(leaky.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- looks amazing")
print(export_text(leaky, feature_names=features + ["trend_pct"]))

The tree just split on `trend_pct` and nailed the label — because the label is **derived from** `trend_pct`. That's **leakage**: the feature is the answer in disguise, and it teaches you nothing.

That's also why the starter data ships **only observable signals** — the product's own decision flags (health scores, "needs CTR fix", and so on) aren't included, so you can't accidentally train on them. You build from what was knowable *before* the outcome.

> Rule of thumb: if a feature would only be known *because someone already made the decision you're predicting*, it leaks. Leave it out.

## 4. 🔧 Your turn
- Change `max_depth` to 3 or 4 — does Precision@50 improve? Can you still read the tree?
- Swap in different features (drop `impressions_90d`, add `engagement_rate`). What does the tree choose to split on first?
- **Important caveat:** we scored *in-sample* here for teaching. The real pipeline uses **client-holdout** validation (`scripts/03_train_model.py`) so a client's pages never appear in both train and test. Re-run your comparison with a train/test split and see if the gap holds.

Write your experiment below.

In [ ]:
# My experiment — all three parts of "your turn".

# ---- Part 1: does a deeper tree (more questions) score better? ----
# max_depth = how many yes/no questions the tree may ask in a row.
for depth in (2, 3, 4):
    t = DecisionTreeClassifier(max_depth=depth, class_weight="balanced", random_state=42)
    t.fit(X, y)
    p50 = precision_at_k(t.predict_proba(X)[:, 1], y, 50)
    n_rules = t.get_n_leaves()
    print(f"depth={depth}  Precision@50: {p50:.3f}   (tree has {n_rules} end-branches to read)")

# ---- Part 2: swap features — drop impressions_90d, add engagement_rate ----
# What does the tree grab FIRST when its favorite toy is taken away?
features_b = ["content_age_days", "days_since_last_update", "avg_position",
              "ctr", "word_count", "engagement_rate"]
X_b = df[features_b].replace([np.inf, -np.inf], np.nan).fillna(0)
tree_b = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_b, y)
print("\nTree with swapped features (first line = its first question):")
print(export_text(tree_b, feature_names=features_b))

# ---- Part 3: the honest test — client-holdout split ----
# Train on SOME clients' pages, test on OTHER clients' pages the model never saw,
# exactly like the real pipeline. Does the tree still beat the hand rule?
from sklearn.model_selection import GroupShuffleSplit
splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=df["client_id"]))

tree_h = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree_h.fit(X.iloc[train_idx], y[train_idx])
tree_test_score = tree_h.predict_proba(X.iloc[test_idx])[:, 1]
hand_test_score = df["hand_rule_score"].values[test_idx]
y_test = y[test_idx]

print("Scored ONLY on unseen clients' pages:")
for k in (20, 50):
    hr = precision_at_k(hand_test_score, y_test, k)
    tr = precision_at_k(tree_test_score, y_test, k)
    print(f"  Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

### What the results mean (in plain words)

**Part 1 — more questions did NOT help.** Depth 2 scored 0.740, depth 3 scored 0.720, depth 4 scored 0.680. Think of the tree like a game of 20 Questions: the first couple of questions catch the big, real patterns. Extra questions start "memorizing" quirks of these exact 30,000 pages instead of learning rules that work anywhere — like a student who memorizes last year's test answers instead of learning the subject. Also, depth 4 has 16 end-branches — you *can* still read it, but it stops feeling like one clear rule. **Smaller was both better and easier to read.**

**Part 2 — the tree's first question changed to `avg_position`.** When we took away `impressions_90d`, the tree's very first question became "is the page's average search position worse than ~0.55?" and its second was "is the content younger than ~288 days?" So the tree told us its priorities: *where you rank* matters most, then *how old the content is*. Interestingly, it never used `engagement_rate` — the new toy we gave it wasn't as useful as position.

**Part 3 — the honest test humbled the model.** When we trained on some clients and tested only on *other clients the tree had never seen* (exactly what the real pipeline does), the hand rule won at both spots: Precision@20 was 0.650 vs the tree's 0.400, and Precision@50 was 0.640 vs 0.520. Earlier the tree looked better — but that was when it got tested on the same pages it studied. On truly new clients, the simple human rule ("stale + still visible") traveled better than what the tiny tree learned.

**Observed (directional, this sample only):** a tiny tree fit in-sample looked good but did not transfer across clients, while the hand rule did. That's the whole lesson of this notebook in one line — *the score that counts is the one on data the model never saw.* The real pipeline's random forest beats the baseline under this same client-holdout test because it's a stronger model with proper validation — not because it peeked.

### Save your work
**Colab:** *File → Save a copy in GitHub* (your submission) and *File → Save a copy in Drive*.

You now have the two core reflexes of applied ML: **discover before you model**, and **prefer a model you can read and can't fool**. That's the whole foundation the capstone builds on.